# DeepLabV3 + MaxSig — Injection-Point Ablation

`Deeplab-MaxSig.ipynb` wraps `DeepLabV3MaxSig` (ResNet-50 backbone) with a `MaxSigGatedBlock`
(`y = x + gamma * post(MaxSig(x))`, `gamma` starting at 0) at up to two points:

| location | channels | what it is |
|---|---|---|
| `layer4`   | 2048 | right after the backbone's last stage, i.e. the ASPP input |
| `aspp_out` | 256  | right after ASPP's projection, before the final 3x3/1x1 conv + 16x upsample |

The **default** configuration applies MaxSig at **both** points: `maxsig_at=("layer4", "aspp_out")`.

Since the default already combines two injection points, the most natural reviewer question is
whether **each point contributes individually**, or whether one is carrying the whole effect. This
notebook keeps the default and adds the **two scenarios that directly decompose it**:

1. **Default (unchanged)** — MaxSig at both `layer4` and `aspp_out`
2. **MaxSig at `layer4` only (2048ch, backbone output)** — reviewer framing: *"You apply MaxSig at
   two points together — does the backbone-level gate alone reproduce the improvement, or is the
   ASPP-level gate necessary?"*
3. **MaxSig at `aspp_out` only (256ch, post-ASPP)** — reviewer framing: *"Conversely, does the
   post-ASPP gate alone (without touching the backbone) already capture most of the gain?"*

Each scenario is its own cell and reports only: **best validation Dice**, **best epoch**, and
**test Dice**. For the **default configuration** it additionally reports **GFLOPs** and **FPS**.


In [2]:
# ============================================================
# Config
# ============================================================
ROOT = '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/UnetMaxsig/compsitdataset/'

config = {
    "lr": 1e-4,
    "batch_size": 8,
    "epochs": 100,          # lower this (e.g. 40-60) for a faster 3-run sweep if needed
    "backbone_name": "resnet50",
}

RESULTS_DIR = ROOT + 'deeplab_ablation_ckpts/'


In [3]:
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

from time import perf_counter
from glob import glob

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models.segmentation import deeplabv3_resnet50, deeplabv3_resnet101

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


/home/jupyter-r.yazdi/.local/lib/python3.10/site-packages/albumentations/check_version.py:49: UserWarning: Error fetching version info <urlopen error [Errno 101] Network is unreachable>
  data = fetch_version_info()


device: cuda


## Dataset & dataloaders\n(unchanged from `Deeplab-MaxSig.ipynb`)

In [3]:
class mydataset(Dataset):
    def __init__(self, images_dir, masks_dir, transforms_img=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transforms_img = transforms_img
        self.mask_names = glob(masks_dir + "/*.tif") + glob(masks_dir + "/*.bmp") + glob(masks_dir + "/*.png")

    def __len__(self):
        return len(self.mask_names)

    def __getitem__(self, index):
        mask_name = self.mask_names[index]
        mask = cv2.imread(mask_name, cv2.IMREAD_ANYDEPTH)
        mask = (mask - mask.min()) / (mask.max() - mask.min())
        mask = (mask > 0).astype(np.float32)
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_AREA)

        imgn = mask_name.replace('/masks/', '/images/')
        if imgn.endswith('.tif'):
            if not os.path.exists(imgn):
                imgn = imgn.replace('.tif', '.bmp')

        image = cv2.imread(imgn, cv2.IMREAD_ANYDEPTH)
        image = (image - image.min()) / (image.max() - image.min()).astype(np.float32)
        image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_AREA)

        if self.transforms_img is not None:
            transformed = self.transforms_img(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        return image, mask


In [4]:
batch_size = config['batch_size']

train_transform = A.Compose([A.Resize(256, 256), ToTensorV2()])
val_transform   = A.Compose([A.Resize(256, 256), ToTensorV2()])
test_transform  = A.Compose([A.Resize(256, 256), ToTensorV2()])

train_ds = mydataset(ROOT + 'dataset/train/images/', ROOT + 'dataset/train/masks/', train_transform)
val_ds   = mydataset(ROOT + 'dataset/val/images/',   ROOT + 'dataset/val/masks/',   val_transform)
test_ds  = mydataset(ROOT + 'dataset/test/images/',  ROOT + 'dataset/test/masks/',  test_transform)

train_dataset = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dataset   = DataLoader(val_ds,   batch_size=batch_size, shuffle=True)
test_dataset  = DataLoader(test_ds,  batch_size=batch_size, shuffle=True)

print("train batches:", len(train_dataset), "| val batches:", len(val_dataset), "| test batches:", len(test_dataset))


train batches: 37 | val batches: 8 | test batches: 8


## MaxSig components & DeepLabV3 + MaxSig model\n(unchanged from `Deeplab-MaxSig.ipynb`)

In [6]:
class MaxSigLayerNormalized(nn.Module):
    def __init__(self, ch_in: int, ch_out: int, kernel_size: int, stride: int = 1, use_sigmoid_input: bool = True):
        super().__init__()
        assert kernel_size >= 1
        self.ch_in = ch_in
        self.ch_out = ch_out
        self.kernel_size = kernel_size
        self.stride = stride
        self.use_sigmoid_input = use_sigmoid_input

        self.padding = (kernel_size - 1) // 2
        self.kernel_area = kernel_size ** 2
        self.center_index = self.kernel_area // 2

        self.weight = nn.Parameter(torch.rand(1, 1, self.kernel_area, 1, 1))
        self.bias   = nn.Parameter(torch.zeros(1, 1, self.kernel_area, 1, 1))

        self.weight_center = nn.Parameter(torch.tensor(1.0))
        self.weight_median = nn.Parameter(torch.tensor(1.0))

        self.proj = nn.Conv2d(ch_in, ch_out, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C_in, H, W = x.shape
        assert C_in == self.ch_in, f"Expected {self.ch_in} channels, got {C_in}"

        x_in = torch.sigmoid(x) if self.use_sigmoid_input else x

        x_unfold = F.unfold(x_in, kernel_size=self.kernel_size, padding=self.padding, stride=self.stride)
        _, CK2, L = x_unfold.shape
        assert CK2 == C_in * self.kernel_area

        out_h = (H + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_w = (W + 2 * self.padding - self.kernel_size) // self.stride + 1

        x_unfold = x_unfold.view(B, C_in, self.kernel_area, out_h, out_w)

        w = torch.sigmoid(self.weight)
        b = torch.sigmoid(self.bias)

        D = torch.maximum(w, x_unfold) + b

        mean_stat = torch.mean(D, dim=2)
        median_stat, _ = torch.median(D, dim=2)

        center_values = x_unfold[:, :, self.center_index, :, :]

        out = (self.weight_center * center_values + self.weight_median * median_stat - mean_stat)
        out = self.proj(out)
        return out

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.segmentation import deeplabv3_resnet50, deeplabv3_resnet101

class MaxSigGatedBlock(nn.Module):
    """
    Same gated-injection idea as your DownBlockWithMaxSig in Unet, factored out so it
    can be dropped anywhere in a feature pyramid:
 
        y = x + gamma * post(expand(MaxSig(reduce(x))))
 
    gamma starts at `gamma_init` (default 0.0) so a pretrained backbone's
    behavior is preserved at the start of fine-tuning.
    
    A bottleneck is ALWAYS applied:
        1x1 conv (ch -> bottleneck_ch) -> MaxSig -> 1x1 conv (bottleneck_ch -> ch)
    This prevents massive C^2 projection matrices when ch is large (e.g. 2048).
    """
    def __init__(self, ch, bottleneck_ch, kernel_size=3, gamma_init=0.0, use_sigmoid_input=True):
        super().__init__()
        
        # Always apply the bottleneck
        self.reduce = nn.Conv2d(ch, bottleneck_ch, kernel_size=1, bias=False)
        self.maxsig = MaxSigLayerNormalized(
            ch_in=bottleneck_ch,
            ch_out=bottleneck_ch,
            kernel_size=kernel_size,
            stride=1,
            use_sigmoid_input=use_sigmoid_input,
        )
        self.expand = nn.Conv2d(bottleneck_ch, ch, kernel_size=1, bias=False)
            
        self.post = nn.Sequential(
            nn.BatchNorm2d(ch),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        m = self.reduce(x)
        m = self.maxsig(m)
        m = self.expand(m)
        m = self.post(m)
        return x + self.gamma * m
 
 
class DeepLabV3MaxSig(nn.Module):
    """
    DeepLabV3 (ResNet50/101 backbone) with MaxSig gated blocks at up to two
    points:
 
      - "layer4"   : right after the backbone's last stage (2048 ch),
                     i.e. the ASPP input. Analog of injecting MaxSig at the
                     end of your deepest DownBlockWithMaxSig stage.
      - "aspp_out" : right after ASPP's 256-ch projection, before the final
                     3x3/1x1 conv + 16x upsample. This is the spot most
                     likely to help boundary sharpness, since plain
                     DeepLabV3 (no "+") has no low-level-feature decoder.
 
    Args:
        num_classes: 1 for binary segmentation (BCEWithLogitsLoss), else the
            number of semantic classes.
        backbone_name: "resnet50" or "resnet101".
        in_channels: input image channels. Your dataset is single-channel
            ultrasound, so this defaults to 1; the pretrained 3-channel stem
            conv is adapted by averaging its RGB filters (Krizhevsky-style).
        pretrained_backbone: load ImageNet-pretrained torchvision weights.
        maxsig_at: subset of {"layer4", "aspp_out"}.
        maxsig_kernel: kernel size passed to MaxSigLayerNormalized.
        gamma_init: initial gate value. 0.0 = identical to plain DeepLabV3 at
            the start of training (recommended, especially with a pretrained
            backbone).
        use_sigmoid_input: passed through to MaxSigLayerNormalized.
        maxsig_bottleneck_ch: The channel dimension to always project down to 
            inside the MaxSig block before projecting back up.
    """
    def __init__(
        self,
        num_classes: int = 1,
        backbone_name: str = "resnet50",
        in_channels: int = 1,
        pretrained_backbone: bool = True,
        maxsig_at=("layer4", "aspp_out"),
        maxsig_kernel: int = 3,
        gamma_init: float = 0.0,
        use_sigmoid_input: bool = True,
        maxsig_bottleneck_ch: int = 256,
    ):
        super().__init__()
        assert backbone_name in ("resnet50", "resnet101")
        maxsig_at = set(maxsig_at)
        assert maxsig_at <= {"layer4", "aspp_out"}, \
            f"unknown maxsig_at entries: {maxsig_at - {'layer4', 'aspp_out'}}"
 
        builder = deeplabv3_resnet50 if backbone_name == "resnet50" else deeplabv3_resnet101
        base = builder(
            weights=None,
            weights_backbone="DEFAULT" if pretrained_backbone else None,
            num_classes=num_classes,
            aux_loss=False,
        )
 
        self.backbone = base.backbone      # IntermediateLayerGetter -> {'out': layer4 feats}
        self.classifier = base.classifier  # DeepLabHead: [ASPP, Conv3x3, BN, ReLU, Conv1x1]
 
        # --- adapt the stem conv for non-RGB input ---
        if in_channels != 3:
            old_conv = self.backbone.conv1
            new_conv = nn.Conv2d(
                in_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size, stride=old_conv.stride,
                padding=old_conv.padding, bias=False,
            )
            if pretrained_backbone:
                with torch.no_grad():
                    w = old_conv.weight.mean(dim=1, keepdim=True)   # avg RGB filters
                    new_conv.weight.copy_(w.repeat(1, in_channels, 1, 1))
            self.backbone.conv1 = new_conv
 
        # --- MaxSig gated blocks ---
        # Bottleneck is ALWAYS applied: 
        # layer4: 2048 -> 256 -> 2048
        self.msig_layer4 = (
            MaxSigGatedBlock(2048, maxsig_bottleneck_ch, maxsig_kernel, gamma_init, use_sigmoid_input)
            if "layer4" in maxsig_at else None
        )
        # aspp_out: 256 -> 256 -> 256
        self.msig_aspp_out = (
            MaxSigGatedBlock(256, maxsig_bottleneck_ch, maxsig_kernel, gamma_init, use_sigmoid_input)
            if "aspp_out" in maxsig_at else None
        )
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_shape = x.shape[-2:]
 
        feats = self.backbone(x)["out"]           # (B, 2048, H/16, W/16)
        if self.msig_layer4 is not None:
            feats = self.msig_layer4(feats)
 
        aspp_out = self.classifier[0](feats)       # ASPP -> (B, 256, H/16, W/16)
        if self.msig_aspp_out is not None:
            aspp_out = self.msig_aspp_out(aspp_out)
 
        x = self.classifier[1](aspp_out)            # 3x3 conv
        x = self.classifier[2](x)                    # BatchNorm
        x = self.classifier[3](x)                    # ReLU
        x = self.classifier[4](x)                    # 1x1 conv -> num_classes
 
        x = F.interpolate(x, size=input_shape, mode="bilinear", align_corners=False)
        return x

## Ablation helpers

`run_scenario(run_name, maxsig_at, ...)` builds a fresh `DeepLabV3MaxSig` with MaxSig gates at the
given injection points (`()` = plain DeepLabV3, no MaxSig), trains it, evaluates test Dice with
the best checkpoint, and reports only `best_val_dice`, `best_epoch`, `test_dice`.

`compute_flops_fps(...)` reports GFLOPs (via `thop`) and FPS — used only for the default
configuration.


In [4]:
def run_scenario(run_name, maxsig_at, epochs=config["epochs"], verbose_every=10):
    torch.manual_seed(42)

    model = DeepLabV3MaxSig(
        num_classes=1,
        backbone_name=config["backbone_name"],
        in_channels=1,
        pretrained_backbone=False,
        maxsig_at=maxsig_at,
        maxsig_kernel=3,
        gamma_init=0.0,
        use_sigmoid_input=True,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'])
    scheduler = StepLR(optimizer, step_size=100, gamma=0.83)
    scaler = torch.amp.GradScaler('cuda')

    ckpt_path = os.path.join(RESULTS_DIR, f"best-{run_name}.pt")
    best_valid_dice = 0.0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        for inputs, mask in train_dataset:
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda"):
                outputs = model(inputs.to(device))
                labels = mask.to(device).unsqueeze(1)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

        dice_score = 0.0
        with torch.no_grad():
            model.eval()
            for inputs, mask in val_dataset:
                inputs = inputs.to(device)
                preds = model(inputs)
                labels = mask.to(device).unsqueeze(1)
                pred_sig = torch.sigmoid(preds)
                preds_bin = (pred_sig > 0.5).float()
                intersection = (preds_bin * labels).sum()
                union = (preds_bin + labels).sum()
                dice_score += (2 * intersection / union)

        epoch_dc = float(dice_score / len(val_dataset))
        if best_valid_dice < epoch_dc:
            best_valid_dice = epoch_dc
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)

        if verbose_every and (epoch + 1) % verbose_every == 0:
            print(f"[{run_name}] epoch {epoch+1}/{epochs} | val_dice={epoch_dc:.4f} "
                  f"| best={best_valid_dice:.4f}@{best_epoch+1}")

    # ---- test evaluation with the best checkpoint ----
    model.load_state_dict(torch.load(ckpt_path))
    dice_score = 0.0
    with torch.no_grad():
        model.eval()
        for inputs, mask in test_dataset:
            inputs = inputs.to(device)
            preds = model(inputs)
            labels = mask.to(device).unsqueeze(1)
            pred_sig = torch.sigmoid(preds)
            preds_bin = (pred_sig > 0.5).float()
            intersection = (preds_bin * labels).sum()
            union = (preds_bin + labels).sum()
            dice_score += (2 * intersection / union)
    test_dice = float(dice_score / len(test_dataset))

    result = {
        "run_name": run_name,
        "maxsig_at": tuple(sorted(maxsig_at)),
        "best_val_dice": best_valid_dice,
        "best_epoch": best_epoch + 1,
        "test_dice": test_dice,
    }
    print(f"[{run_name}] DONE | best_val_dice={best_valid_dice:.4f} (epoch {best_epoch+1}) "
          f"| test_dice={test_dice:.4f}")
    return result, model


def compute_flops_fps(model, input_size=(1, 1, 256, 256), n_warmup=10, n_iters=50):
    """Reports GFLOPs (thop) and FPS (measured wall-clock throughput) for a model."""
    model.eval()
    dummy = torch.randn(*input_size).to(device)

    try:
        from thop import profile
        macs, params = profile(model, inputs=(dummy,), verbose=False)
        gflops = macs * 2 / 1e9
    except ImportError:
        print("`thop` not installed. Run `pip install thop` to get GFLOPs. Skipping GFLOPs.")
        gflops = None

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = perf_counter()
        for _ in range(n_iters):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = perf_counter()

    fps = n_iters / (t1 - t0)
    return gflops, fps


ablation_results = []


### Scenario 1 — Default: MaxSig at both `layer4` and `aspp_out`\n(the configuration already used in `Deeplab-MaxSig.ipynb`)

In [7]:
res, default_model = run_scenario(run_name="default_layer4_aspp_out", maxsig_at=("layer4", "aspp_out"))
ablation_results.append(res)


/tmp/ipykernel_880033/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[default_layer4_aspp_out] epoch 10/100 | val_dice=0.6112 | best=0.6545@9
[default_layer4_aspp_out] epoch 20/100 | val_dice=0.7201 | best=0.7231@16
[default_layer4_aspp_out] epoch 30/100 | val_dice=0.7252 | best=0.7351@24
[default_layer4_aspp_out] epoch 40/100 | val_dice=0.7307 | best=0.7471@33
[default_layer4_aspp_out] epoch 50/100 | val_dice=0.7389 | best=0.7502@44
[default_layer4_aspp_out] epoch 60/100 | val_dice=0.7548 | best=0.7554@53
[default_layer4_aspp_out] epoch 70/100 | val_dice=0.7460 | best=0.7554@53
[default_layer4_aspp_out] epoch 80/100 | val_dice=0.7431 | best=0.7554@53
[default_layer4_aspp_out] epoch 90/100 | val_dice=0.7478 | best=0.7679@83
[default_layer4_aspp_out] epoch 100/100 | val_dice=0.7626 | best=0.7679@83
[default_layer4_aspp_out] DONE | best_val_dice=0.7679 (epoch 83) | test_dice=0.7343


### Scenario 2 — MaxSig at **`layer4` only (2048ch, backbone output)**
> Reviewer framing: *"You apply MaxSig at two points together — does the backbone-level gate
> alone reproduce the improvement, or is the ASPP-level gate necessary?"*


In [8]:
res, _ = run_scenario(run_name="layer4_only", maxsig_at=("layer4",))
ablation_results.append(res)


/tmp/ipykernel_880033/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[layer4_only] epoch 10/100 | val_dice=0.7299 | best=0.7299@10
[layer4_only] epoch 20/100 | val_dice=0.7228 | best=0.7299@10
[layer4_only] epoch 30/100 | val_dice=0.6851 | best=0.7559@27
[layer4_only] epoch 40/100 | val_dice=0.7421 | best=0.7559@27
[layer4_only] epoch 50/100 | val_dice=0.7649 | best=0.7649@50
[layer4_only] epoch 60/100 | val_dice=0.7613 | best=0.7649@50
[layer4_only] epoch 70/100 | val_dice=0.7612 | best=0.7649@50
[layer4_only] epoch 80/100 | val_dice=0.7608 | best=0.7813@73
[layer4_only] epoch 90/100 | val_dice=0.7531 | best=0.7813@73
[layer4_only] epoch 100/100 | val_dice=0.7449 | best=0.7813@73
[layer4_only] DONE | best_val_dice=0.7813 (epoch 73) | test_dice=0.7301


### Scenario 3 — MaxSig at **`aspp_out` only (256ch, post-ASPP)**
> Reviewer framing: *"Conversely, does the post-ASPP gate alone (without touching the backbone)
> already capture most of the gain?"*


In [9]:
res, _ = run_scenario(run_name="aspp_out_only", maxsig_at=("aspp_out",))
ablation_results.append(res)


/tmp/ipykernel_880033/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[aspp_out_only] epoch 10/100 | val_dice=0.7031 | best=0.7031@10
[aspp_out_only] epoch 20/100 | val_dice=0.7403 | best=0.7403@20
[aspp_out_only] epoch 30/100 | val_dice=0.7280 | best=0.7403@20
[aspp_out_only] epoch 40/100 | val_dice=0.7339 | best=0.7472@32
[aspp_out_only] epoch 50/100 | val_dice=0.7326 | best=0.7491@48
[aspp_out_only] epoch 60/100 | val_dice=0.7157 | best=0.7503@54
[aspp_out_only] epoch 70/100 | val_dice=0.7432 | best=0.7561@69
[aspp_out_only] epoch 80/100 | val_dice=0.7611 | best=0.7611@80
[aspp_out_only] epoch 90/100 | val_dice=0.7382 | best=0.7739@85
[aspp_out_only] epoch 100/100 | val_dice=0.7564 | best=0.7739@85
[aspp_out_only] DONE | best_val_dice=0.7739 (epoch 85) | test_dice=0.7418


## GFLOPs / FPS — default configuration only\n(`layer4` + `aspp_out`, as used in `Deeplab-MaxSig.ipynb`)

In [10]:
gflops, fps = compute_flops_fps(default_model, input_size=(1, 1, 256, 256))
print(f"Default config (layer4+aspp_out) | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")


Default config (layer4+aspp_out) | GFLOPs: 84.558 | FPS: 122.54


In [7]:
model = DeepLabV3MaxSig(
        num_classes=1,
        backbone_name=config["backbone_name"],
        in_channels=1,
        pretrained_backbone=False,
        maxsig_at=(),
        maxsig_kernel=3,
        gamma_init=0.0,
        use_sigmoid_input=True,
    ).to(device)
gflops, fps = compute_flops_fps(model, input_size=(1, 1, 256, 256))
print(f"baseline model | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")


baseline model | GFLOPs: 81.854 | FPS: 172.49


## Summary table

In [11]:
df = pd.DataFrame(ablation_results)
df = df[["run_name", "maxsig_at", "best_val_dice", "best_epoch", "test_dice"]]
df.loc[df["run_name"] == "default_layer4_aspp_out", "GFLOPs"] = gflops
df.loc[df["run_name"] == "default_layer4_aspp_out", "FPS"] = fps
df


,run_name,maxsig_at,best_val_dice,best_epoch,test_dice,GFLOPs,FPS
0,default_layer4_aspp_out,"(aspp_out, layer4)",0.767898,83,0.734266,84.557699,122.53639
1,layer4_only,"(layer4,)",0.781336,73,0.730139,NaN,NaN
2,aspp_out_only,"(aspp_out,)",0.773934,85,0.741849,NaN,NaN
